# NumPy数组上的计算：通用函数


到目前为止，我们讨论了NumPy的一些基本概念。在接下来的几章中，我们将深入探讨NumPy在Python数据科学领域的重要性，主要是因为它提供了一个简单灵活的接口，以优化对数据数组的计算。

在NumPy数组上的计算可以非常快速，也可能非常缓慢。

使其快速的关键是使用向量化操作，这通常通过NumPy的*通用函数*（ufuncs）来实现。

本章阐述了NumPy ufuncs 的必要性，它们可以用于提高对数组元素进行重复计算的效率。

接下来介绍了NumPy包中许多最常见和有用的算术ufuncs。

## 循环的缓慢性

Python 的默认实现（称为 CPython）在某些操作上执行得非常缓慢。

这部分是由于语言的动态和解释性质；类型灵活，因此操作序列无法像 C 和 Fortran 等语言那样编译成高效的机器代码。

最近有多种尝试来解决这一弱点：著名的例子包括 [PyPy 项目](http://pypy.org/)，这是一个即时编译的 Python 实现；[Cython 项目](http://cython.org)，它将 Python 代码转换为可编译的 C 代码；以及 [Numba 项目](http://numba.pydata.org/)，它将 Python 代码片段转换为快速 LLVM 字节码。

这些项目各有优缺点，但可以肯定地说，这三种方法都尚未超越标准 CPython 引擎的影响力和受欢迎程度。

Python 的相对迟缓通常表现为许多小操作被重复执行时，例如，循环遍历数组以对每个元素进行操作。

例如，假设我们有一个值数组，我们希望计算每个值的倒数。

一种简单的方法可能如下所示：

In [1]:
import numpy as np
rng = np.random.default_rng(seed=1701)

def compute_reciprocals(values):
    output = np.empty(len(values))
    for i in range(len(values)):
        output[i] = 1.0 / values[i]
    return output
        
values = rng.integers(1, 10, size=5)
compute_reciprocals(values)

array([0.11111111, 0.25      , 1.        , 0.33333333, 0.125     ])

这个实现对于来自C或Java背景的人来说，可能感觉相当自然。

然而，如果我们对这段代码进行大输入的执行时间测量，就会发现这个操作非常慢——也许令人惊讶的是如此之慢！

我们将使用IPython的`%timeit`魔法命令来进行基准测试（在[Profiling and Timing Code](01.07-Timing-and-Profiling.ipynb)中讨论）。

In [2]:
big_array = rng.integers(1, 100, size=1000000)
%timeit compute_reciprocals(big_array)

522 ms ± 5.69 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


计算这百万次操作并存储结果需要几秒钟的时间！

即使是手机，其处理速度也以千亿次浮点运算（即每秒数十亿次数值操作）来衡量，这似乎显得异常缓慢。

事实证明，瓶颈不在于操作本身，而在于CPython在循环的每个周期中必须进行的类型检查和函数调度。

每当计算倒数时，Python首先会检查对象的类型，并动态查找适用于该类型的正确函数。

如果我们使用编译代码，那么这种类型规范将在代码执行之前就已确定，从而可以更高效地计算结果。

## 引入 Ufuncs

对于许多类型的操作，NumPy 提供了一个方便的接口，用于这种静态类型编译例程。这被称为 *向量化* 操作。

对于像这里的逐元素除法这样的简单操作，向量化就像直接在数组对象上使用 Python 算术运算符一样简单。

这种向量化方法旨在将循环推送到 NumPy 底层的编译层，从而实现更快的执行速度。

请比较以下两个操作的结果：

In [3]:
print(compute_reciprocals(values))
print(1.0 / values)

[0.11111111 0.25       1.         0.33333333 0.125     ]
[0.11111111 0.25       1.         0.33333333 0.125     ]


查看我们大数组的执行时间，可以发现其完成速度比Python循环快几个数量级。

In [4]:
%timeit (1.0 / big_array)

530 μs ± 14.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


在NumPy中，向量化操作是通过ufuncs实现的，其主要目的是快速执行对NumPy数组中值的重复操作。

Ufuncs具有极大的灵活性——之前我们看到过标量与数组之间的操作，但我们也可以在两个数组之间进行操作。

In [5]:
np.arange(5) / np.arange(1, 6)

array([0.        , 0.5       , 0.66666667, 0.75      , 0.8       ])

ufunc 操作不仅限于一维数组，它们也可以作用于多维数组。

In [6]:
x = np.arange(9).reshape((3, 3))
2 ** x

array([[  1,   2,   4],
       [  8,  16,  32],
       [ 64, 128, 256]])

使用ufuncs进行向量化计算通常比使用Python循环实现的对应方法更高效，尤其是在数组增大时。

每当您在NumPy脚本中看到这样的循环时，应考虑是否可以用向量化表达式替代。

## 探索 NumPy 的 Ufuncs

Ufuncs 有两种类型：*一元 ufuncs*，它们对单个输入进行操作，以及 *二元 ufuncs*，它们对两个输入进行操作。

在这里，我们将看到这两种类型函数的示例。

### 数组运算

NumPy 的 ufuncs 使用 Python 的原生算术运算符，使得使用起来非常自然。

标准的加法、减法、乘法和除法都可以使用：

In [7]:
x = np.arange(4)
print("x      =", x)
print("x + 5  =", x + 5)
print("x - 5  =", x - 5)
print("x * 2  =", x * 2)
print("x / 2  =", x / 2)
print("x // 2 =", x // 2)  # floor division

x      = [0 1 2 3]
x + 5  = [5 6 7 8]
x - 5  = [-5 -4 -3 -2]
x * 2  = [0 2 4 6]
x / 2  = [0.  0.5 1.  1.5]
x // 2 = [0 0 1 1]


还有一个用于取反的单目ufunc，一个用于指数运算的`**`运算符，以及一个用于求模的`%`运算符：

In [8]:
print("-x     = ", -x)
print("x ** 2 = ", x ** 2)
print("x % 2  = ", x % 2)

-x     =  [ 0 -1 -2 -3]
x ** 2 =  [0 1 4 9]
x % 2  =  [0 1 0 1]


此外，这些可以根据您的需要任意串联，并且遵循标准的运算顺序。

In [9]:
-(0.5*x + 1) ** 2

array([-1.  , -2.25, -4.  , -6.25])

所有这些算术操作只是围绕NumPy内置的特定ufuncs的方便封装。例如，`+`运算符是`add` ufunc的一个封装。

In [10]:
np.add(x, 2)

array([2, 3, 4, 5])

以下表格列出了在NumPy中实现的算术运算符：

| Operator    | Equivalent ufunc  | Description                         |
|-------------|-------------------|-------------------------------------|
|`+`          |`np.add`           |Addition (e.g., `1 + 1 = 2`)         |
|`-`          |`np.subtract`      |Subtraction (e.g., `3 - 2 = 1`)      |
|`-`          |`np.negative`      |Unary negation (e.g., `-2`)          |
|`*`          |`np.multiply`      |Multiplication (e.g., `2 * 3 = 6`)   |
|`/`          |`np.divide`        |Division (e.g., `3 / 2 = 1.5`)       |
|`//`         |`np.floor_divide`  |Floor division (e.g., `3 // 2 = 1`)  |
|`**`         |`np.power`         |Exponentiation (e.g., `2 ** 3 = 8`)  |
|`%`          |`np.mod`           |Modulus/remainder (e.g., `9 % 4 = 1`)|

此外，还有布尔/按位运算符；我们将在[比较、掩码和布尔逻辑](02.06-Boolean-Arrays-and-Masks.ipynb)中探讨这些内容。

### 绝对值

正如 NumPy 理解 Python 内置的算术运算符一样，它也理解 Python 的内置绝对值函数：

In [10]:
x = np.array([-2, -1, 0, 1, 2])
abs(x)

array([2, 1, 0, 1, 2])

相应的 NumPy ufunc 是 `np.absolute`，也可以通过别名 `np.abs` 访问。

In [11]:
np.absolute(x)

array([2, 1, 0, 1, 2])

In [12]:
np.abs(x)

array([2, 1, 0, 1, 2])

该ufunc也可以处理复数数据，在这种情况下，它返回幅度：

In [13]:
x = np.array([3 - 4j, 4 - 3j, 2 + 0j, 0 + 1j])
np.abs(x)

array([5., 5., 2., 1.])

### 三角函数

NumPy 提供了大量有用的 ufuncs，其中一些对数据科学家来说尤其重要的是三角函数。

我们将首先定义一个角度数组：

In [14]:
theta = np.linspace(0, np.pi, 3)

现在我们可以对这些值计算一些三角函数：

In [15]:
print("theta      = ", theta)
print("sin(theta) = ", np.sin(theta))
print("cos(theta) = ", np.cos(theta))
print("tan(theta) = ", np.tan(theta))

theta      =  [0.         1.57079633 3.14159265]
sin(theta) =  [0.0000000e+00 1.0000000e+00 1.2246468e-16]
cos(theta) =  [ 1.000000e+00  6.123234e-17 -1.000000e+00]
tan(theta) =  [ 0.00000000e+00  1.63312394e+16 -1.22464680e-16]


计算结果达到机器精度，因此应为零的值并不总是恰好等于零。

反三角函数也可用：

In [16]:
x = [-1, 0, 1]
print("x         = ", x)
print("arcsin(x) = ", np.arcsin(x))
print("arccos(x) = ", np.arccos(x))
print("arctan(x) = ", np.arctan(x))

x         =  [-1, 0, 1]
arcsin(x) =  [-1.57079633  0.          1.57079633]
arccos(x) =  [3.14159265 1.57079633 0.        ]
arctan(x) =  [-0.78539816  0.          0.78539816]


### 指数与对数

在NumPy的ufuncs中，还有其他常见的操作，即指数运算。

In [17]:
x = [1, 2, 3]
print("x   =", x)
print("e^x =", np.exp(x))
print("2^x =", np.exp2(x))
print("3^x =", np.power(3., x))

x   = [1, 2, 3]
e^x = [ 2.71828183  7.3890561  20.08553692]
2^x = [2. 4. 8.]
3^x = [ 3.  9. 27.]


指数的逆运算是对数，也可以使用。

基本的 `np.log` 函数提供自然对数；如果您希望计算以2为底或以10为底的对数，这些也可以使用。

In [18]:
x = [1, 2, 4, 10]
print("x        =", x)
print("ln(x)    =", np.log(x))
print("log2(x)  =", np.log2(x))
print("log10(x) =", np.log10(x))

x        = [1, 2, 4, 10]
ln(x)    = [0.         0.69314718 1.38629436 2.30258509]
log2(x)  = [0.         1.         2.         3.32192809]
log10(x) = [0.         0.30103    0.60205999 1.        ]


也有一些专门的版本，对于保持非常小输入的精度是很有用的。

In [19]:
x = [0, 0.001, 0.01, 0.1]
print("exp(x) - 1 =", np.expm1(x))
print("log(1 + x) =", np.log1p(x))

exp(x) - 1 = [0.         0.0010005  0.01005017 0.10517092]
log(1 + x) = [0.         0.0009995  0.00995033 0.09531018]


当 `x` 非常小时，这些函数提供的值比直接使用 `np.log` 或 `np.exp` 更加精确。

### 专用 Ufuncs

NumPy 提供了许多其他的 ufunc，包括双曲三角函数、按位算术运算、比较操作、弧度与度数之间的转换、舍入和余数等功能。

浏览 NumPy 文档可以发现许多有趣的功能。

另一个优秀的专用 ufunc 来源是子模块 `scipy.special`。

如果您想在数据上计算某个冷门数学函数，可能它已经在 `scipy.special` 中实现。

可供列举的函数实在太多，但以下代码片段展示了一些可能出现在统计学上下文中的函数。

In [20]:
from scipy import special

In [21]:
# Gamma functions (generalized factorials) and related functions
x = [1, 5, 10]
print("gamma(x)     =", special.gamma(x))
print("ln|gamma(x)| =", special.gammaln(x))
print("beta(x, 2)   =", special.beta(x, 2))

gamma(x)     = [1.0000e+00 2.4000e+01 3.6288e+05]
ln|gamma(x)| = [ 0.          3.17805383 12.80182748]
beta(x, 2)   = [0.5        0.03333333 0.00909091]


In [22]:
# Error function (integral of Gaussian),
# its complement, and its inverse
x = np.array([0, 0.3, 0.7, 1.0])
print("erf(x)  =", special.erf(x))
print("erfc(x) =", special.erfc(x))
print("erfinv(x) =", special.erfinv(x))

erf(x)  = [0.         0.32862676 0.67780119 0.84270079]
erfc(x) = [1.         0.67137324 0.32219881 0.15729921]
erfinv(x) = [0.         0.27246271 0.73286908        inf]


在NumPy和`scipy.special`中，还有许多其他的ufunc可用。

由于这些软件包的文档可以在线获取，因此进行类似于“gamma函数 python”的网络搜索通常会找到相关信息。

## 高级 Ufunc 特性

许多 NumPy 用户在使用 ufunc 时，从未了解其完整的特性。

在这里，我将概述一些 ufunc 的特殊功能。

### 指定输出

对于大型计算，有时指定结果存储的数组是非常有用的。

对于所有的ufuncs，可以使用函数的`out`参数来实现这一点。

In [23]:
x = np.arange(5)
y = np.empty(5)
np.multiply(x, 10, out=y)
print(y)

[ 0. 10. 20. 30. 40.]


这甚至可以与数组视图一起使用。例如，我们可以将计算结果写入指定数组的每个其他元素中：

In [24]:
y = np.zeros(10)
np.power(2, x, out=y[::2])
print(y)

[ 1.  0.  2.  0.  4.  0.  8.  0. 16.  0.]


如果我们改为写成 `y[::2] = 2 ** x`，这将导致创建一个临时数组来存储 `2 ** x` 的结果，然后再进行第二次操作，将这些值复制到 `y` 数组中。

对于如此小的计算，这并没有太大区别，但对于非常大的数组来说，谨慎使用 `out` 参数所带来的内存节省可能是显著的。

### 聚合

对于二元ufunc，可以直接从对象计算聚合。

例如，如果我们想用特定操作对数组进行*归约*，可以使用任何ufunc的`reduce`方法。

归约会重复地将给定操作应用于数组的元素，直到只剩下一个结果。

例如，对`add` ufunc调用`reduce`返回数组中所有元素的总和。

In [25]:
x = np.arange(1, 6)
np.add.reduce(x)

np.int64(15)

同样，调用 `reduce` 函数于 `multiply` ufunc 会得到所有数组元素的乘积：

In [26]:
np.multiply.reduce(x)

np.int64(120)

如果我们希望存储计算的所有中间结果，可以改用 `accumulate`：

In [27]:
np.add.accumulate(x)

array([ 1,  3,  6, 10, 15])

In [28]:
np.multiply.accumulate(x)

array([  1,   2,   6,  24, 120])

请注意，对于这些特定情况，有专门的NumPy函数用于计算结果（`np.sum`、`np.prod`、`np.cumsum`、`np.cumprod`），我们将在[聚合：最小值、最大值及其间的一切](02.04-Computation-on-arrays-aggregates.ipynb)中进行探讨。

### 外积

最后，任何 ufunc 都可以使用 `outer` 方法计算两个不同输入的所有配对输出。

这使得您可以在一行代码中完成诸如创建乘法表等操作。

In [29]:
x = np.arange(1, 6)
np.multiply.outer(x, x)

array([[ 1,  2,  3,  4,  5],
       [ 2,  4,  6,  8, 10],
       [ 3,  6,  9, 12, 15],
       [ 4,  8, 12, 16, 20],
       [ 5, 10, 15, 20, 25]])

`ufunc.at` 和 `ufunc.reduceat` 方法同样非常有用，我们将在 [Fancy Indexing](02.07-Fancy-Indexing.ipynb) 中进行探讨。

我们还将遇到 ufunc 能够在不同形状和大小的数组之间操作的能力，这组操作被称为 *广播*。

这个主题的重要性足以让我们专门花一整章来讨论它（见 [Computation on Arrays: Broadcasting](02.05-Computation-on-arrays-broadcasting.ipynb)）。

## Ufuncs: 进一步学习


有关通用函数的更多信息（包括可用函数的完整列表）可以在 [NumPy](http://www.numpy.org) 和 [SciPy](http://www.scipy.org) 的文档网站上找到。

请记住，您还可以通过导入这些包并使用 IPython 的自动补全和帮助功能（`?`），直接从 IPython 中访问相关信息，具体说明见 [IPython中的帮助与文档](01.01-Help-And-Documentation.ipynb)。